In [ ]:
import pandas as pd
import numpy as np

# 假设 df 是上一段程序输出的 final_df
df = final_df.copy()

# 确保数据按股票和时间正确排序，这对于计算动量和滞后极其重要
df = df.sort_values(['permno', 'date'])

# ---------------------------------------------------------
# 1. 规模特征 (Size)
# ---------------------------------------------------------
# 习惯上取对数处理极值
df['Size'] = np.log(df['me'])

# ---------------------------------------------------------
# 2. 价值特征 (Value - Book-to-Market)
# ---------------------------------------------------------
# 账面所有者权益 (ceq) / 市值 (me)
df['BM'] = df['ceq'] / df['me']
# 处理可能出现的负账面价值或极值
df['BM'] = np.where(df['BM'] < 0, np.nan, df['BM']) 

# ---------------------------------------------------------
# 3. 盈利能力特征 (Profitability - Operating Profitability)
# ---------------------------------------------------------
# 营业利润 = 收入(revt) - 销货成本(cogs) - 销售及管理费用(xsga) - 利息支出(xint)
# 缩放变量通常使用账面权益(ceq)或总资产(at)
df['OP'] = (df['revt'] - df['cogs'].fillna(0) - df['xsga'].fillna(0) - df['xint'].fillna(0)) / df['ceq']

# ---------------------------------------------------------
# 4. 投资特征 (Investment - Asset Growth)
# ---------------------------------------------------------
# 总资产增长率：(当期总资产 - 上期总资产) / 上期总资产
df['at_lag1'] = df.groupby('permno')['at'].shift(1)
df['INV'] = (df['at'] - df['at_lag1']) / df['at_lag1']

# ---------------------------------------------------------
# 5. 短期反转特征 (Short-term Reversal - MOM1m)
# ---------------------------------------------------------
# 过去1个月的收益率 (t-1)
df['MOM1m'] = df.groupby('permno')['ret'].shift(1)

# ---------------------------------------------------------
# 6. 动量特征 (Momentum - MOM12m)
# ---------------------------------------------------------
# 经典定义：过去12个月的累计收益率，但跳过最近1个月 (即 t-12 到 t-2)
# 先计算对数收益率以便于连加计算累计收益
df['log_ret'] = np.log(1 + df['ret'])
# 滚动求和11个月（t-12 到 t-2）
df['MOM12m'] = df.groupby('permno')['log_ret'].apply(
    lambda x: x.shift(2).rolling(window=11).sum()
).reset_index(level=0, drop=True)
# 转回普通收益率
df['MOM12m'] = np.exp(df['MOM12m']) - 1

# ---------------------------------------------------------
# 截面标准化 (Cross-sectional Rank Normalization) - 极度重要！
# ---------------------------------------------------------
# 神经网络对输入特征的尺度非常敏感。在横截面定价中，我们通常将每个月的特征映射到 [-1, 1] 区间
features = ['Size', 'BM', 'OP', 'INV', 'MOM1m', 'MOM12m']

def cs_normalize(group):
    # 使用 Rank 转化可以完美消除财务数据中的极端异常值（Outliers）
    return (group.rank() - 1) / (len(group) - 1) * 2 - 1

for feat in features:
    df[f'{feat}_norm'] = df.groupby('date')[feat].transform(cs_normalize)